<a href="https://colab.research.google.com/github/0IBSIP/0IBSIP/blob/main/Copy_of_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# TASK 3 — DATA CLEANING
# Dataset: Titanic Train Dataset
# ============================================================

import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# Load dataset
df = pd.read_csv("train.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
# ============================================================
# DATA QUALITY REPORT — BEFORE CLEANING
# ============================================================

quality_report = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str),
    "Null_Count": df.isnull().sum(),
    "Null_Percentage": (df.isnull().mean() * 100).round(2),
    "Unique_Values": df.nunique()
}).reset_index(drop=True)

print("========== DATA QUALITY REPORT ==========")

display(quality_report)

print("\n========== DATASET SUMMARY ==========")
print("Total Rows       :", df.shape[0])
print("Total Columns    :", df.shape[1])
print("Total Null Values:", df.isnull().sum().sum())
print("Duplicate Rows   :", df.duplicated().sum())

========== DATA QUALITY REPORT ==========


,Column,Data_Type,Null_Count,Null_Percentage,Unique_Values
0,PassengerId,int64,0,0.00,891
1,Survived,int64,0,0.00,2
2,Pclass,int64,0,0.00,3
3,Name,object,0,0.00,891
4,Sex,object,0,0.00,2
5,Age,float64,177,19.87,88
6,SibSp,int64,0,0.00,7
7,Parch,int64,0,0.00,7
8,Ticket,object,0,0.00,681
9,Fare,float64,0,0.00,248



========== DATASET SUMMARY ==========
Total Rows       : 891
Total Columns    : 12
Total Null Values: 866
Duplicate Rows   : 0


In [ ]:
# ============================================================
# NUMERIC RANGE REPORT
# ============================================================

numeric_cols = df.select_dtypes(include=np.number).columns

range_report = pd.DataFrame({
    "Column": numeric_cols,
    "Minimum": [df[col].min() for col in numeric_cols],
    "Maximum": [df[col].max() for col in numeric_cols],
    "Mean": [df[col].mean() for col in numeric_cols],
    "Median": [df[col].median() for col in numeric_cols],
    "Std_Dev": [df[col].std() for col in numeric_cols]
})

print("========== NUMERIC RANGE REPORT ==========")
display(range_report)

========== NUMERIC RANGE REPORT ==========


,Column,Minimum,Maximum,Mean,Median,Std_Dev
0,PassengerId,1.00,891.0000,446.000000,446.0000,257.353842
1,Survived,0.00,1.0000,0.383838,0.0000,0.486592
2,Pclass,1.00,3.0000,2.308642,3.0000,0.836071
3,Age,0.42,80.0000,29.699118,28.0000,14.526497
4,SibSp,0.00,8.0000,0.523008,0.0000,1.102743
5,Parch,0.00,6.0000,0.381594,0.0000,0.806057
6,Fare,0.00,512.3292,32.204208,14.4542,49.693429


In [ ]:
# ============================================================
# SAVE BEFORE-CLEANING METRICS
# ============================================================

before_rows = len(df)
before_columns = len(df.columns)
before_nulls = df.isnull().sum().sum()
before_duplicates = df.duplicated().sum()

print("========== BEFORE CLEANING ==========")
print("Rows:", before_rows)
print("Columns:", before_columns)
print("Null Values:", before_nulls)
print("Duplicate Rows:", before_duplicates)

========== BEFORE CLEANING ==========
Rows: 891
Columns: 12
Null Values: 866
Duplicate Rows: 0


In [ ]:
# ============================================================
# DUPLICATE REMOVAL
# ============================================================

duplicate_count = df.duplicated().sum()

print("Duplicate rows found:", duplicate_count)

df = df.drop_duplicates().reset_index(drop=True)

print("Duplicate rows removed:", duplicate_count)
print("Rows after duplicate removal:", len(df))

Duplicate rows found: 0
Duplicate rows removed: 0
Rows after duplicate removal: 891


In [ ]:
# ============================================================
# STANDARDIZE COLUMN NAMES
# ============================================================

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

print("Standardized column names:")
print(df.columns.tolist())

Standardized column names:
['passengerid', 'survived', 'pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked']


In [ ]:

# ============================================================
# STANDARDIZE TEXT VALUES
# ============================================================

text_columns = ["name", "sex", "ticket", "cabin", "embarked"]

for col in text_columns:
    df[col] = df[col].astype("string").str.strip()

# Standardize Sex
df["sex"] = (
    df["sex"]
    .str.lower()
    .replace({
        "m": "male",
        "f": "female"
    })
)

# Standardize Embarked
df["embarked"] = (
    df["embarked"]
    .str.upper()
)

print("Text standardization completed.")

Text standardization completed.


In [ ]:
# ============================================================
# MISSING VALUE HANDLING
# ============================================================

print("Missing values BEFORE treatment:")
display(df.isnull().sum())

# Age → Median
age_median = df["age"].median()
df["age"] = df["age"].fillna(age_median)

# Embarked → Mode
embarked_mode = df["embarked"].mode()[0]
df["embarked"] = df["embarked"].fillna(embarked_mode)

# Cabin → Unknown
# Missing Cabin information is retained as a meaningful category.
df["cabin"] = df["cabin"].fillna("Unknown")

print("\nMissing values AFTER treatment:")
display(df.isnull().sum())

Missing values BEFORE treatment:


,0
passengerid,0
survived,0
pclass,0
name,0
sex,0
age,177
sibsp,0
parch,0
ticket,0
fare,0



Missing values AFTER treatment:


,0
passengerid,0
survived,0
pclass,0
name,0
sex,0
age,0
sibsp,0
parch,0
ticket,0
fare,0


## Missing Data Handling Decisions

### Age
177 missing values were found in the Age column.

Median imputation was selected because Age is a numerical variable
and the median is less affected by extreme values than the mean.

### Embarked
Only 2 values were missing in Embarked.

Mode imputation was selected because Embarked is a categorical
variable and the most frequent category provides a reasonable
replacement for the small number of missing observations.

### Cabin
687 values were missing in Cabin.

Instead of deleting the entire column or rows, missing Cabin values
were replaced with "Unknown". Missing cabin information itself can
carry useful information and replacing it with a separate category
preserves the observations.

### PassengerId
PassengerId is an identifier, so statistical imputation was not
applied to this column because there were no missing values.

### Other Columns
The remaining columns contained no missing values and therefore
required no missing-data treatment.

In [ ]:
# ============================================================
# DATA TYPE CORRECTION
# ============================================================

# PassengerId is an identifier → string
df["passengerid"] = df["passengerid"].astype("string")

# Text / categorical columns
df["name"] = df["name"].astype("string")
df["sex"] = df["sex"].astype("string")
df["ticket"] = df["ticket"].astype("string")
df["cabin"] = df["cabin"].astype("string")
df["embarked"] = df["embarked"].astype("string")

# Numeric columns
df["survived"] = pd.to_numeric(df["survived"], errors="coerce")
df["pclass"] = pd.to_numeric(df["pclass"], errors="coerce")
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["sibsp"] = pd.to_numeric(df["sibsp"], errors="coerce")
df["parch"] = pd.to_numeric(df["parch"], errors="coerce")
df["fare"] = pd.to_numeric(df["fare"], errors="coerce")

print("Data types corrected successfully.")

display(df.dtypes.to_frame("Final Data Type"))

Data types corrected successfully.


,Final Data Type
passengerid,string[python]
survived,int64
pclass,int64
name,string[python]
sex,string[python]
age,float64
sibsp,int64
parch,int64
ticket,string[python]
fare,float64


In [ ]:
# ============================================================
# OUTLIER DETECTION — IQR METHOD
# ============================================================

numeric_columns = [
    "age",
    "sibsp",
    "parch",
    "fare"
]

outlier_results = []

for col in numeric_columns:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_mask = (
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    )

    outlier_count = outlier_mask.sum()

    outlier_results.append({
        "Column": col,
        "Q1": round(Q1, 2),
        "Q3": round(Q3, 2),
        "IQR": round(IQR, 2),
        "Lower_Bound": round(lower_bound, 2),
        "Upper_Bound": round(upper_bound, 2),
        "Outlier_Count": int(outlier_count),
        "Outlier_Percentage":
            round(outlier_count / len(df) * 100, 2)
    })

outlier_report = pd.DataFrame(outlier_results)

print("========== IQR OUTLIER REPORT ==========")
display(outlier_report)

========== IQR OUTLIER REPORT ==========


,Column,Q1,Q3,IQR,Lower_Bound,Upper_Bound,Outlier_Count,Outlier_Percentage
0,age,22.00,35.0,13.00,2.50,54.50,66,7.41
1,sibsp,0.00,1.0,1.00,-1.50,2.50,46,5.16
2,parch,0.00,0.0,0.00,0.00,0.00,213,23.91
3,fare,7.91,31.0,23.09,-26.72,65.63,116,13.02


## Outlier Treatment Decision

The IQR method was used to identify statistical outliers.

Detected outliers were not automatically deleted because extreme
values in Titanic variables can represent legitimate observations.

For example, high Fare values may correspond to passengers who
purchased expensive tickets, while higher Age values can represent
genuine older passengers.

Therefore, the detected outliers were retained rather than removed.
This avoids unnecessary information loss.

The outlier analysis is documented so that downstream analysts can
make an informed decision depending on their modelling objective.

In [ ]:
# ============================================================
# LOGICAL / RANGE VALIDATION
# ============================================================

print("========== LOGICAL VALIDATION ==========")

# Age should not be negative
print("Negative Age values:", (df["age"] < 0).sum())

# Fare should not be negative
print("Negative Fare values:", (df["fare"] < 0).sum())

# Pclass should be between 1 and 3
print(
    "Invalid Pclass values:",
    (~df["pclass"].isin([1, 2, 3])).sum()
)

# Survived should be 0 or 1
print(
    "Invalid Survived values:",
    (~df["survived"].isin([0, 1])).sum()
)

# Embarked should contain C, Q, S
print(
    "Invalid Embarked values:",
    (~df["embarked"].isin(["C", "Q", "S"])).sum()
)

========== LOGICAL VALIDATION ==========
Negative Age values: 0
Negative Fare values: 0
Invalid Pclass values: 0
Invalid Survived values: 0
Invalid Embarked values: 0


In [ ]:
# ============================================================
# FINAL NULL CHECK
# ============================================================

final_nulls = df.isnull().sum()

final_null_report = pd.DataFrame({
    "Column": df.columns,
    "Null_Count": final_nulls.values,
    "Null_Percentage":
        (final_nulls / len(df) * 100).round(2).values
})

print("========== FINAL NULL REPORT ==========")
display(final_null_report)

print("Total remaining null values:", final_nulls.sum())

========== FINAL NULL REPORT ==========


,Column,Null_Count,Null_Percentage
0,passengerid,0,0.0
1,survived,0,0.0
2,pclass,0,0.0
3,name,0,0.0
4,sex,0,0.0
5,age,0,0.0
6,sibsp,0,0.0
7,parch,0,0.0
8,ticket,0,0.0
9,fare,0,0.0


Total remaining null values: 0


In [ ]:
# ============================================================
# FINAL DUPLICATE CHECK
# ============================================================

final_duplicates = df.duplicated().sum()

print("========== FINAL DUPLICATE CHECK ==========")
print("Remaining duplicate rows:", final_duplicates)

========== FINAL DUPLICATE CHECK ==========
Remaining duplicate rows: 0


In [ ]:
# ============================================================
# BEFORE VS AFTER SUMMARY
# ============================================================

after_rows = len(df)
after_columns = len(df.columns)
after_nulls = df.isnull().sum().sum()
after_duplicates = df.duplicated().sum()

summary = pd.DataFrame({
    "Metric": [
        "Row Count",
        "Column Count",
        "Total Null Values",
        "Duplicate Rows"
    ],

    "Before Cleaning": [
        before_rows,
        before_columns,
        before_nulls,
        before_duplicates
    ],

    "After Cleaning": [
        after_rows,
        after_columns,
        after_nulls,
        after_duplicates
    ]
})

print("========== BEFORE VS AFTER ==========")
display(summary)

========== BEFORE VS AFTER ==========


,Metric,Before Cleaning,After Cleaning
0,Row Count,891,891
1,Column Count,12,12
2,Total Null Values,866,0
3,Duplicate Rows,0,0


In [ ]:
# ============================================================
# FINAL DATA QUALITY REPORT
# ============================================================

final_quality_report = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Null_Count": df.isnull().sum().values,
    "Unique_Values": df.nunique().values
})

print("========== FINAL DATA QUALITY REPORT ==========")

display(final_quality_report)

========== FINAL DATA QUALITY REPORT ==========


,Column,Data_Type,Null_Count,Unique_Values
0,passengerid,string,0,891
1,survived,int64,0,2
2,pclass,int64,0,3
3,name,string,0,891
4,sex,string,0,2
5,age,float64,0,88
6,sibsp,int64,0,7
7,parch,int64,0,7
8,ticket,string,0,681
9,fare,float64,0,248


In [ ]:
# ============================================================
# CLEANED DATASET PREVIEW
# ============================================================

print("========== CLEANED DATASET ==========")

print("Shape:", df.shape)

display(df.head(10))

========== CLEANED DATASET ==========
Shape: (891, 12)


,passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,Unknown,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,Unknown,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,Unknown,S
5,6,0,3,"Moran, Mr. James",male,28.0,0,0,330877,8.4583,Unknown,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,Unknown,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,Unknown,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,Unknown,C


In [ ]:
# ============================================================
# SAVE CLEANED DATASET
# ============================================================

output_file = "Titanic_Cleaned_Task3.csv"

df.to_csv(
    output_file,
    index=False
)

print("Cleaned dataset saved successfully!")
print("File:", output_file)

Cleaned dataset saved successfully!
File: Titanic_Cleaned_Task3.csv


In [ ]:
# ============================================================
# DOWNLOAD CLEANED DATASET
# ============================================================

from google.colab import files

files.download("Titanic_Cleaned_Task3.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Task 3 — Data Cleaning Conclusion

The Titanic dataset was cleaned using a structured and reproducible
Python/Pandas workflow.

### Cleaning activities completed

- Generated an initial data quality report.
- Identified missing values in Age, Cabin and Embarked.
- Applied median imputation to Age.
- Applied mode imputation to Embarked.
- Replaced missing Cabin values with "Unknown".
- Checked and removed duplicate records.
- Standardized text and categorical values.
- Converted PassengerId to string because it is an identifier.
- Corrected numerical and categorical data types.
- Detected numerical outliers using the IQR method.
- Retained legitimate statistical outliers to avoid unnecessary
  information loss.
- Performed logical range validation.
- Generated a before-vs-after data quality summary.
- Exported the final analysis-ready dataset to CSV.

The cleaned dataset is now suitable for exploratory data analysis,
visualization and machine-learning workflows.